# 03b · Bangladesh check set v2 (2019–2023): Sen1Floods11 models, fine-tuning, Otsu

Same protocol as notebook 03, on the larger v2 check set. Notebook 02 trained Sentinel-1 water U-Nets on
[Sen1Floods11](https://github.com/cloudtostreet/Sen1Floods11); this notebook scores them on **323 Bangladesh chips**, then
fine-tunes on Bangladesh chips with every chip scored by a model that never saw its tile (or its date).

**Check set** (release asset `bd_s1_water_checkset_v2.tar.gz`): 358 chips of 512 × 512 px, July–October 2019–2023.
* Radar and label exactly as in v1 (same chipping code; only the search windows differ). Radar and optical scenes 4.5–19.5 h apart.
* 35 chips were rejected **only for optical defects** (21 unmasked cumulus, 11 cloud shadow labelled water, 3 haze with
  cumulus). The decisions were made by a vision-language model on contact sheets and spot-checked by a second model,
  not by a person (see `README_QC.md` in the tarball). 323 are kept.

**Baseline:** Otsu threshold on VV, fitted per chip.

Five times more chips than v1, so fine-tuning takes about five times longer than notebook 03.

In [ ]:
!pip -q install segmentation-models-pytorch rasterio scikit-image
import os, hashlib, time, urllib.request, tarfile, numpy as np, pandas as pd, rasterio, torch, torch.nn.functional as F
import segmentation_models_pytorch as smp
from scipy.ndimage import uniform_filter
from skimage.filters import threshold_otsu
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(torch.cuda.get_device_name(0) if dev == 'cuda' else 'CPU only')

In [ ]:
# Release assets, checked against their sha256
REL = 'https://github.com/deluair/bangladesh-water-ml/releases/download/v0.1.0'
REL_SET = 'https://github.com/deluair/bangladesh-water-ml/releases/download/v0.2.0'   # check set v2 is a v0.2.0 asset; weights stay in v0.1.0
ASSETS = {
    'bd_s1_water_checkset_v2.tar.gz': '01482b9479f9d846ab57cb0c7baf8b82cc4cf7d191616a880edd9563f88449de',
    'unet_s1_water.pt': 'b91dfdbfd8b00c8dcdbfd909cc5fd0657d230a3e3b378a39653084896dd11631',
    'unet_s1_water_robust_s20260927.pt': '63a67c3dac7b48c8ef2a013f3d8a8091c05552d1182142e89d1db4759ac44037',
    'unet_s1_water_robust_s1.pt': '6da558652150d6ceff160647611dec9294d4fb9f0c1490a6704d38de85ccde45',
    'unet_s1_water_robust_s2.pt': '5889a11b011c2491ba02c766bc761d074068503382baea05a213c7b8b74a16b4',
}
for f, h in ASSETS.items():
    if not os.path.exists(f): urllib.request.urlretrieve((REL_SET if f.endswith('.tar.gz') else REL) + '/' + f, f)
    assert hashlib.sha256(open(f, 'rb').read()).hexdigest() == h, f
if not os.path.exists('bd/index.csv'):
    os.makedirs('bd', exist_ok=True); tarfile.open('bd_s1_water_checkset_v2.tar.gz').extractall('bd')
q = pd.read_csv('bd/qc_decisions.csv')
print(q.decision.value_counts().to_dict()); print(q[q.decision != 'keep'].reason.value_counts().to_dict())

In [ ]:
# Kept chips; normalisation = Sen1Floods11 hand-labelled train split (notebook 02)
MU = np.array([-11.188685, -17.898247], 'float32'); SD = np.array([6.83887, 6.5755134], 'float32')
q = q[q.decision == 'keep'].reset_index(drop=True)
def rd(p):
    with rasterio.open(p) as d: return d.read()
X = np.stack([np.nan_to_num(np.clip(rd(f'bd/chips/{i}_S1.tif').astype('float32'), -50, 5), nan=-50.0) for i in q.id])
Y = np.stack([rd(f'bd/chips/{i}_Label.tif')[0].astype('int64') for i in q.id])
nrm = lambda x: (x - MU[:, None, None]) / SD[:, None, None]
print(X.shape, 'water share of labelled pixels', round(float((Y == 1).sum() / (Y >= 0).sum()), 3))

def predict(model, idx):
    model.eval(); out = []
    with torch.no_grad():
        for i in idx: out.append((model(torch.from_numpy(nrm(X[i])[None]).to(dev))[0, 0] > 0).cpu().numpy())
    return out
def pooled(preds, idx):
    I = U = 0
    for p, i in zip(preds, idx):
        v = Y[i] >= 0; t = Y[i] == 1; I += (p & t & v).sum(); U += ((p | t) & v).sum()
    return I / U
oI = oU = 0
for i in range(len(q)):
    vv = X[i][0]; v = (Y[i] >= 0) & (vv > -50); t = Y[i] == 1; po = (vv < threshold_otsu(vv[v])) & v
    oI += (po & t & v).sum(); oU += ((po | t) & v).sum()
OTSU = oI / oU; print(f'Otsu (per-chip VV threshold) pooled IoU {OTSU:.3f}')

In [ ]:
# 1. Sen1Floods11 models applied to Bangladesh as they are
def load(f):
    m = smp.Unet('resnet34', encoder_weights=None, in_channels=2, classes=1)
    m.load_state_dict(torch.load(f, map_location='cpu')); return m.to(dev)
allidx = np.arange(len(q)); rows = []
for f in ['unet_s1_water.pt'] + [f'unet_s1_water_robust_s{s}.pt' for s in (20260927, 1, 2)]:
    rows.append(dict(model=f, bd_pooled_iou=round(pooled(predict(load(f), allidx), allidx), 3)))
zero_shot = pd.DataFrame(rows); zero_shot.loc[len(zero_shot)] = ['Otsu baseline', round(OTSU, 3)]; zero_shot

In [ ]:
# 2. Fine-tune on Bangladesh chips, 2-fold grouped split: each chip is scored by a model that never saw its group
def aug(x, y):
    med = np.median(x, axis=(1, 2), keepdims=True)
    x = (x - med) * np.random.uniform(0.75, 1.25, (2, 1, 1)) + med + np.random.uniform(-5, 5, (2, 1, 1))
    if np.random.rand() < .3: x = 10 * np.log10(np.maximum(uniform_filter(10 ** (x / 10), (1, 3, 3)), 1e-6))
    k = np.random.randint(4); x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k)
    if np.random.rand() < .5: x, y = x[:, :, ::-1], y[:, ::-1]
    return x.astype('float32').copy(), y.copy()
def loss_fn(lg, y):
    m = (y >= 0).float(); t = y.clamp(min=0).float(); lg = lg[:, 0]
    bce = (F.binary_cross_entropy_with_logits(lg, t, reduction='none') * m).sum() / m.sum().clamp(min=1)
    p = torch.sigmoid(lg) * m; return bce + 1 - (2 * (p * t).sum() + 1) / (p.sum() + (t * m).sum() + 1)
def folds(split):
    g = q.id.str.extract(r'_(T\d\d[A-Z]{3})_')[0] if split == 'tile' else q.id.str[3:11]   # S2 tile, or S2 date
    fold, load_ = {}, [0, 0]
    for k, n in g.value_counts().items(): j = int(np.argmin(load_)); fold[k] = j; load_[j] += n   # greedy balance
    return g.map(fold).values, g.nunique()
res = []
for split in ('tile', 'date'):
    fd, ng = folds(split)
    for init in ('s1f11', 'imagenet'):
        for seed in (20260927, 1, 2):
            preds, idxs = [], []
            for f in (0, 1):
                torch.manual_seed(seed); np.random.seed(seed)
                tr, te = np.where(fd != f)[0], np.where(fd == f)[0]
                m = (smp.Unet('resnet34', encoder_weights='imagenet', in_channels=2, classes=1).to(dev) if init == 'imagenet'
                     else load(f'unet_s1_water_robust_s{seed}.pt'))
                opt = torch.optim.AdamW(m.parameters(), 1e-4, weight_decay=1e-4)
                for ep in range(25):
                    m.train(); perm = np.random.permutation(tr)
                    for b in range(0, len(perm), 4):
                        xb, yb = zip(*[aug(X[i], Y[i]) for i in perm[b:b + 4]])
                        l = loss_fn(m(torch.from_numpy(nrm(np.stack(xb))).to(dev)), torch.from_numpy(np.stack(yb)).to(dev))
                        opt.zero_grad(); l.backward(); opt.step()
                preds += predict(m, te); idxs += list(te)
            res.append(dict(split=split, groups=ng, init=init, seed=seed, iou=pooled(preds, idxs)))
            print(res[-1])
res = pd.DataFrame(res); res.to_csv('bd_finetune_colab.csv', index=False)

In [ ]:
summary = res.groupby(['split', 'init']).iou.agg(['mean', 'std', 'count']).round(3)
summary['otsu'] = round(OTSU, 3); summary

**Reading the result.** On check set v2 (release v0.2.0, run with `a2_bd/bd_finetune.py` on an Apple M5 Max GPU),
the Sen1Floods11 models score 0.266 / 0.161 / 0.106 as trained, against 0.472 for Otsu. Fine-tuning on about 161 Bangladesh
chips per fold lifts the U-Net to 0.799–0.807, and the encoder's starting point (Sen1Floods11 or ImageNet) changes the result
by at most 0.003. Scores from this notebook on a CUDA GPU will differ slightly, because GPU arithmetic is not bit-identical
across hardware.